# 04 — Customer Feature Engineering

Build monthly customer snapshots using a **180-day lookback** and a **90-day forward label horizon**. Features must use only information available before the snapshot date. A customer is labelled positive if they make at least one valid purchase in the next 90 days.

The reusable implementation is in `src/features.py`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "features.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import DB_FILE
from src.features import (
    build_customer_snapshots, FEATURES_FILE, LOOKBACK_DAYS,
    HORIZON_DAYS, NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET
)
import duckdb

print("Lookback:", LOOKBACK_DAYS, "days | Label horizon:", HORIZON_DAYS, "days")

Lookback: 180 days | Label horizon: 90 days


## 1. Load cleaned transactions

In [2]:
if not DB_FILE.exists():
    raise FileNotFoundError("Run Notebook 02 (or python -m src.data) first.")
with duckdb.connect(str(DB_FILE), read_only=True) as con:
    transactions = con.execute("SELECT * FROM transactions").fetchdf()
print("Transaction rows:", len(transactions))

Transaction rows: 1033036


## 2. Build snapshots

In [3]:
snapshots = build_customer_snapshots(transactions)
print("Snapshot rows:", len(snapshots))
print("Snapshot dates:", snapshots.snapshot_date.min(), "to", snapshots.snapshot_date.max())
display(snapshots.head())

Snapshot rows: 48079
Snapshot dates: 2010-06-01 00:00:00 to 2011-09-01 00:00:00


,customer_id,invoice_count_180d,monetary_180d,units_180d,distinct_products_180d,active_days_180d,country,avg_invoice_value_180d,recency_days,repeat_purchase_90d,snapshot_date
0,12346,10,230.55,51,7,6,United Kingdom,23.0550,91,1,2010-06-01
1,12349,2,1268.52,474,47,2,Italy,634.2600,14,0,2010-06-01
2,12355,1,488.21,303,22,1,Bahrain,488.2100,11,0,2010-06-01
3,12358,1,1429.83,309,17,1,Austria,1429.8300,175,1,2010-06-01
4,12359,4,1522.23,777,65,4,Cyprus,380.5575,70,1,2010-06-01


## 3. Inspect target balance and feature distributions

In [4]:
display(snapshots.groupby("snapshot_date")[TARGET].agg(["count", "mean"]).rename(columns={"mean":"repeat_purchase_rate"}))
display(snapshots[NUMERIC_FEATURES].describe().T)
display(snapshots[TARGET].value_counts(normalize=True).rename("share").to_frame())

,count,repeat_purchase_rate
snapshot_date,,
2010-06-01,2669,0.499063
2010-07-01,2757,0.507436
2010-08-01,2822,0.569454
2010-09-01,2831,0.619216
2010-10-01,2910,0.578007
2010-11-01,3209,0.492365
2010-12-01,3479,0.379419
2011-01-01,3415,0.376574
2011-02-01,3359,0.396249


,count,mean,std,min,25%,50%,75%,max
recency_days,48079.0,65.720252,50.564722,1.0,22.000,53.00,104.00,180.00
invoice_count_180d,48079.0,2.867031,4.370181,1.0,1.000,2.00,3.00,124.00
monetary_180d,48079.0,1309.432346,4689.533814,1.3,264.150,521.64,1139.07,179248.38
avg_invoice_value_180d,48079.0,399.144969,1003.145887,1.3,177.855,295.09,436.10,77183.60
units_180d,48079.0,802.454356,3601.033874,1.0,129.000,288.00,656.00,220600.00
distinct_products_180d,48079.0,45.408432,58.625576,1.0,14.000,29.00,56.00,1245.00
active_days_180d,48079.0,2.560203,3.274164,1.0,1.000,2.00,3.00,76.00


,share
repeat_purchase_90d,
0,0.50937
1,0.49063


## 4. Persist the modelling table

In [5]:
FEATURES_FILE.parent.mkdir(parents=True, exist_ok=True)
snapshots.to_parquet(FEATURES_FILE, index=False)
print("Saved:", FEATURES_FILE)

Saved: /Users/nithiii/Downloads/data-science-ai-portfolio/01-machine-learning/customer-retention-revenue-intelligence/data/processed/customer_snapshots.parquet


## Leakage and target checks

- Confirm every feature is derived from dates strictly before its snapshot.
- Confirm the target uses purchases from the snapshot date up to (but not including) snapshot date + 90 days.
- Customers with no purchase in the lookback are not represented in this dataset; the model estimates repeat purchase among customers with recent historical activity.
- Inspect class balance over time before modelling.